In [1]:
using Plots
using Distributions, Random, Statistics
using LinearAlgebra, SparseArrays, JSON
using Colors
using CSV
using DataFrames
using MLJ
using Dates
using JuMP, Ipopt
using LinearAlgebra

purple = RGB(122/256, 30/256, 71/256)
blue   = RGB(0/256, 39/256, 76/256)
mygreen = RGB(83/256, 165/256, 132/256)

In [11]:
# ---------- Load and combine data ----------

dfs = [CSV.read("location$i.csv", DataFrame; types=Dict(:Time => String)) for i in 1:4]

for (i, data) in enumerate(dfs)
    data.Time = DateTime.(data.Time, dateformat"yyyy-mm-dd HH:MM:SS")
    data.LocationID = fill(i, nrow(data))
end

df = vcat(dfs...)
sort!(df, [:LocationID, :Time])

m = nrow(df)
times = df.Time
t = Float64.(df.Power)
location_ids = df.LocationID


# ---------- Generate lagged power ----------

lag_power = fill(NaN, m)

for i in 2:m
    if location_ids[i] == location_ids[i-1] &&
       times[i] - times[i-1] == Hour(1)

        lag_power[i] = t[i-1]
    end
end


# ---------- Create 24-hour groups ----------

hours_per_group = 24
groups = Vector{Int}[]

for loc in unique(location_ids)
    loc_idx = findall(==(loc), location_ids)

    for i in 1:hours_per_group:(length(loc_idx) - hours_per_group + 1)
        idx = loc_idx[i:i + hours_per_group - 1]

        if all(times[idx[j]] - times[idx[j-1]] == Hour(1) for j in 2:hours_per_group)
            push!(groups, idx)
        end
    end
end

num_groups = length(groups)


# ---------- Partition groups chronologically ----------

train, val, test = partition(1:num_groups, 0.8, 0.1; shuffle=false, rng=120)

train_idx = vcat(groups[train]...)
val_idx   = vcat(groups[val]...)
test_idx  = vcat(groups[test]...)


# ---------- Filter observations with valid lagged power ----------

train_idx_fit = filter(i -> !isnan(lag_power[i]), train_idx)
val_idx_fit   = filter(i -> !isnan(lag_power[i]), val_idx)
test_idx_fit  = filter(i -> !isnan(lag_power[i]), test_idx)


# ---------- Normalize weather features using training data ----------

max_10m  = maximum(Float64.(df.windspeed_10m[train_idx_fit]))
max_100m = maximum(Float64.(df.windspeed_100m[train_idx_fit]))
max_10g  = maximum(Float64.(df.windgusts_10m[train_idx_fit]))

x10m  = Float64.(df.windspeed_10m) ./ max_10m
x100m = Float64.(df.windspeed_100m) ./ max_100m
x10g  = Float64.(df.windgusts_10m) ./ max_10g;

In [5]:
# 2.
# Define RBF functions

c = collect(0.0:0.01:1)
num_centers = length(c)

function rbf_features(xs, γ)

    Φ = zeros(length(xs), num_centers)

    for i in eachindex(xs), j in 1:num_centers
        Φ[i, j] = exp(-(γ * abs(xs[i] - c[j]))^2)
    end

    return Φ
end

weather_design_matrix(x1, x2, x3, γ) = hcat(rbf_features(x1, γ),rbf_features(x2, γ),rbf_features(x3, γ))
grouped_design_matrix(x1, x2, x3, lag_power, γ) = hcat(weather_design_matrix(x1, x2, x3, γ),lag_power)

Y_all = t;

In [15]:
# 3. 
# Training loop

include("market_functions.jl")

function market_cost(P_pred, P_actual)

    P_predicted = P_pred * 50.0
    Pg_DA, cost_DA = day_ahead_dispatch(600.0 - P_predicted)
    _, _, cost_RT = real_time_dispatch(Pg_DA,600.0 - P_actual;kup = 1.0,kdown = 1.0)

    return cost_DA + cost_RT
end

function train_rbf(Φ, t, actual_power, Gamma;α = 1e-7,n_epochs = 15,λ = 0.001,ρ = 1e-5,μ = 0.0,reg_type = :l2,seed = 128)

    rng = MersenneTwister(seed)

    ntrain = size(Φ, 1)
    nfeat = size(Φ, 2)

    w_0 = 0.0
    w = zeros(nfeat)

    for epoch in 1:n_epochs

        losses = Float64[]

        for i in shuffle(rng, 1:ntrain)

            # RBF prediction
            P_pred = w_0 + dot(Φ[i, :], w)

            # Actual power in MW
            P_actual = actual_power[i]

            # Market cost
            cost_prediction = market_cost(P_pred, P_actual)

            # Regression loss
            loss = (t[i] - P_pred)^2
            push!(losses, loss)

            # Market cost gradient
            epsilon = 1e-4

            dC_dP = (
                market_cost(P_pred + epsilon, P_actual) -
                market_cost(P_pred - epsilon, P_actual)
            ) / (2 * epsilon)

            # Regression loss gradient
            dL_dP = 2 * (P_pred - t[i])

            # Gradients with respect to RBF weights
            grad_cost_w0 = dC_dP
            grad_cost_w = dC_dP .* Φ[i, :]

            grad_loss_w0 = dL_dP
            grad_loss_w = dL_dP .* Φ[i, :]

            # Combined gradients
            grad_w0 = grad_cost_w0 + μ * grad_loss_w0
            grad_w = grad_cost_w + μ * grad_loss_w

            # Update RBF parameters
            w_0 -= α * grad_w0
            w .-= α .* grad_w

            # Weight regularization
            if reg_type == :l2
                w .-= α * λ .* w

            elseif reg_type == :l1
                w .= sign.(w) .* max.(0, abs.(w) .- α * λ)
            end
        end

        # Average regression loss
        average_loss = mean(losses)

        # Update Lagrange multiplier
        μ = max(0.0, μ + ρ * (average_loss - Gamma))

        println("Epoch $epoch | Average loss: $average_loss | μ: $μ")
    end

    return w_0, w
end

predict_rbf(Φ, w_0, w) = w_0 .+ Φ * w;

In [18]:
# Generate training RBF features
γ = 5.0
Gamma = 0.05

Phi_train = grouped_design_matrix(x10m[train_idx_fit],x100m[train_idx_fit],x10g[train_idx_fit],lag_power[train_idx_fit],γ)

Y_train = t[train_idx_fit]

actual_power_train = Y_train .* 50.0

# ---------- Small DFL test subset ----------

n_small = 750

Phi_small = Phi_train[1:n_small, :]
Y_small = Y_train[1:n_small]
actual_power_small = actual_power_train[1:n_small]

# Train DFL model
w0_test, w_test = train_rbf(Phi_small,Y_small,actual_power_small,Gamma;n_epochs = 5)

# Generate predictions
Y_small_pred = predict_rbf(Phi_small, w0_test, w_test)

# Evaluate regression error
small_RMSE = sqrt(mean((Y_small .- Y_small_pred).^2))

println("Small subset testing RMSE: ", small_RMSE)

Epoch 1 | Average loss: 0.22972730818985065 | μ: 1.7972730818985067e-6
Epoch 2 | Average loss: 0.1389423559230472 | μ: 2.686696641128979e-6
Epoch 3 | Average loss: 0.11402236178876612 | μ: 3.32692025901664e-6
Epoch 4 | Average loss: 0.11219086135486124 | μ: 3.948828872565253e-6
Epoch 5 | Average loss: 0.11101332411959011 | μ: 4.558962113761154e-6
Small subset testing RMSE: 0.3216296957187243
